In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import os
import shutil
# 安装必要的库
!pip install torch torchvision

# 下载数据集
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz

# 解压数据集
!tar -xvf images.tar.gz
!tar -xvf annotations.tar.gz

# 检查解压后的文件夹结构
import os
print(os.listdir('images'))
print(os.listdir('annotations'))
# 数据增强和预处理
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

data_dir = 'images'
annotations_file = 'annotations/list.txt'

cat_breeds = [
    "Abyssinian", "Bengal", "Birman", "Bombay", "British_Shorthair", "Egyptian_Mau",
    "Maine_Coon", "Persian", "Ragdoll", "Russian_Blue", "Siamese", "Sphynx",
    "American_Bobtail", "American_Curl", "American_Shorthair", "American_Wirehair",
    "Balinese", "Chartreux", "Cornish_Rex", "Devon_Rex", "Havana_Brown",
    "Himalayan", "Japanese_Bobtail", "Korat", "LaPerm", "Manx", "Nebelung",
    "Ocicat", "Oriental", "Pixiebob", "Savannah", "Scottish_Fold", "Selkirk_Rex",
    "Siberian", "Snowshoe", "Somali", "Tonkinese", "Turkish_Angora", "Turkish_Van"
]

# 创建每个猫品种的文件夹
with open(annotations_file) as f:
    lines = f.readlines()

for line in lines:
    line = line.strip()
    if line.startswith("#"):
        continue
    parts = line.split()
    file_name = parts[0]
    breed_id = int(parts[1]) - 1
    if breed_id < len(cat_breeds):
        breed = cat_breeds[breed_id]
        img_path = os.path.join(data_dir, file_name + '.jpg')
        breed_folder = os.path.join('organized_cats', breed)

        if not os.path.exists(breed_folder):
            os.makedirs(breed_folder)

        shutil.move(img_path, os.path.join(breed_folder, file_name + '.jpg'))

# 加载数据集
full_dataset = datasets.ImageFolder('organized_cats', transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

import torch.nn as nn
import torch.optim as optim
from torchvision import models

# 定义模型
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.cnn = models.resnet18(pretrained=True)
        num_ftrs = self.cnn.fc.in_features
        self.cnn.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        return x

num_classes = len(cat_breeds)
model = CNNModel(num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# 训练模型
num_epochs = 20
best_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)

    print(f'Epoch {epoch}/{num_epochs - 1}, Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}')

    model.eval()
    val_running_corrects = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            val_running_corrects += torch.sum(preds == labels.data)

    val_acc = val_running_corrects.double() / len(val_dataset)
    print(f'Validation Acc: {val_acc:.4f}')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_cat_breed_cnn.pth')

    scheduler.step()

print(f'Best Validation Acc: {best_acc:.4f}')

# 加载最佳模型
model.load_state_dict(torch.load('best_cat_breed_cnn.pth'))
model.eval()

# 预测新图像
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()

# 读取并处理照片
for fn in uploaded.keys():
    img = Image.open(fn)
    img = transform(img).unsqueeze(0).to(device)

    # 预测
    with torch.no_grad():
        output = model(img)
        _, predicted = torch.max(output, 1)
        breed = cat_breeds[predicted.item()]

    print(f"The predicted breed is: {breed}")

    # 显示照片
    img = Image.open(fn)
    plt.imshow(img)
    plt.axis('off')
    plt.show()
